In [3]:
import os
import math
import numpy as np
import pandas as pd
import numba as nb


# ----------------------------
# Numba: binary heap helpers
# ----------------------------
@nb.njit(cache=True)
def _heap_push_min(hp: np.ndarray, hi: np.ndarray, size: int, val: np.float32, idx: int) -> int:
    i = size
    size += 1
    while i > 0:
        p = (i - 1) // 2
        if hp[p] <= val:
            break
        hp[i] = hp[p]
        hi[i] = hi[p]
        i = p
    hp[i] = val
    hi[i] = idx
    return size


@nb.njit(cache=True)
def _heap_pop_min(hp: np.ndarray, hi: np.ndarray, size: int) -> tuple[np.float32, int, int]:
    # returns (val, idx, new_size)
    val = hp[0]
    idx = hi[0]
    size -= 1
    if size > 0:
        lastv = hp[size]
        lasti = hi[size]
        i = 0
        while True:
            l = 2 * i + 1
            if l >= size:
                break
            r = l + 1
            c = l
            if r < size and hp[r] < hp[l]:
                c = r
            if hp[c] >= lastv:
                break
            hp[i] = hp[c]
            hi[i] = hi[c]
            i = c
        hp[i] = lastv
        hi[i] = lasti
    return val, idx, size


# ----------------------------
# Numba: hit-time kernels (1D output per level)
# ----------------------------
@nb.njit(cache=True)
def _hit_long_tp_1level(open_f32, high_f32, mult_tp, out_u32, heap_p, heap_i):
    T = open_f32.shape[0]
    INF = np.uint32(T)
    for i in range(T):
        out_u32[i] = INF

    size = 0
    for t in range(T):
        size = _heap_push_min(heap_p, heap_i, size, np.float32(open_f32[t]), t)
        thr = np.float32(high_f32[t] / mult_tp)  # open <= high/(1+tp)
        while size > 0 and heap_p[0] <= thr:
            _, idx, size = _heap_pop_min(heap_p, heap_i, size)
            out_u32[idx] = np.uint32(t)


@nb.njit(cache=True)
def _hit_long_sl_1level(open_f32, low_f32, mult_down, out_u32, heap_p, heap_i):
    # long SL: low <= open*(1-sl)  <=> open >= low/(1-sl)
    # use max-heap via storing neg_open in min-heap
    T = open_f32.shape[0]
    INF = np.uint32(T)
    for i in range(T):
        out_u32[i] = INF

    size = 0
    for t in range(T):
        size = _heap_push_min(heap_p, heap_i, size, np.float32(-open_f32[t]), t)
        thr_open = np.float32(low_f32[t] / mult_down)    # mult_down = (1-sl)
        thr_neg = np.float32(-thr_open)
        while size > 0 and heap_p[0] <= thr_neg:
            _, idx, size = _heap_pop_min(heap_p, heap_i, size)
            out_u32[idx] = np.uint32(t)


@nb.njit(cache=True)
def _hit_short_tp_1level(open_f32, low_f32, mult_down, out_u32, heap_p, heap_i):
    # short TP: profit on drop: low <= open*(1-tp) <=> open >= low/(1-tp)
    # max-heap via -open
    T = open_f32.shape[0]
    INF = np.uint32(T)
    for i in range(T):
        out_u32[i] = INF

    size = 0
    for t in range(T):
        size = _heap_push_min(heap_p, heap_i, size, np.float32(-open_f32[t]), t)
        thr_open = np.float32(low_f32[t] / mult_down)  # mult_down = (1-tp)
        thr_neg = np.float32(-thr_open)
        while size > 0 and heap_p[0] <= thr_neg:
            _, idx, size = _heap_pop_min(heap_p, heap_i, size)
            out_u32[idx] = np.uint32(t)


@nb.njit(cache=True)
def _hit_short_sl_1level(open_f32, high_f32, mult_up, out_u32, heap_p, heap_i):
    # short SL: high >= open*(1+sl) <=> open <= high/(1+sl)
    T = open_f32.shape[0]
    INF = np.uint32(T)
    for i in range(T):
        out_u32[i] = INF

    size = 0
    for t in range(T):
        size = _heap_push_min(heap_p, heap_i, size, np.float32(open_f32[t]), t)
        thr = np.float32(high_f32[t] / mult_up)
        while size > 0 and heap_p[0] <= thr:
            _, idx, size = _heap_pop_min(heap_p, heap_i, size)
            out_u32[idx] = np.uint32(t)


def _pct_grid(start_pct: float, stop_pct: float, step_pct: float) -> np.ndarray:
    # inclusive stop (с допуском)
    eps = 1e-9
    g = np.arange(start_pct, stop_pct + eps, step_pct, dtype=np.float32)
    if g.size == 0:
        raise ValueError("Empty grid")
    return g


def precompute_hit_times_execution_1m(
    data: np.ndarray | pd.DataFrame,
    *,
    open_col: int | str = "open",
    high_col: int | str = "high",
    low_col: int | str = "low",
    tp_start_pct: float,
    tp_stop_pct: float,
    tp_step_pct: float,
    sl_start_pct: float,
    sl_stop_pct: float,
    sl_step_pct: float,
    out_dir: str | None = None,
    prefix: str = "hit_time",
    use_memmap: bool = True,
) -> dict:
    """
    Precompute hit-time tables for execution timeframe 1m.

    Entry assumption:
      entry at open[t] (the bar t itself can trigger TP/SL via high/low[t]).

    Output convention:
      hit_* arrays have shape (n_levels, T) and dtype uint32.
      value = first bar index where level is touched, or INF = T if never.

    If out_dir is provided and use_memmap=True:
      arrays are created as .npy-backed memmaps (fast load later via mmap_mode="r").
    """
    # --- extract arrays ---
    if isinstance(data, pd.DataFrame):
        open_ = data[open_col].to_numpy(dtype=np.float32, copy=False)
        high_ = data[high_col].to_numpy(dtype=np.float32, copy=False)
        low_ = data[low_col].to_numpy(dtype=np.float32, copy=False)
    else:
        arr = np.asarray(data)
        if arr.ndim != 2:
            raise ValueError("data must be 2D (T, M)")
        if isinstance(open_col, str) or isinstance(high_col, str) or isinstance(low_col, str):
            raise ValueError("If data is ndarray, open_col/high_col/low_col must be int indices.")
        open_ = arr[:, int(open_col)].astype(np.float32, copy=False)
        high_ = arr[:, int(high_col)].astype(np.float32, copy=False)
        low_ = arr[:, int(low_col)].astype(np.float32, copy=False)

    T = open_.shape[0]
    if T < 2:
        raise ValueError("Need at least 2 bars")

    # --- grids ---
    tp_grid_pct = _pct_grid(tp_start_pct, tp_stop_pct, tp_step_pct)   # in %
    sl_grid_pct = _pct_grid(sl_start_pct, sl_stop_pct, sl_step_pct)   # in %

    tp = tp_grid_pct / np.float32(100.0)
    sl = sl_grid_pct / np.float32(100.0)

    tp_mult_up = (np.float32(1.0) + tp).astype(np.float32)        # 1+tp
    tp_mult_down = (np.float32(1.0) - tp).astype(np.float32)      # 1-tp
    sl_mult_up = (np.float32(1.0) + sl).astype(np.float32)        # 1+sl
    sl_mult_down = (np.float32(1.0) - sl).astype(np.float32)      # 1-sl

    n_tp = tp.size
    n_sl = sl.size

    # --- alloc helper (store as (n_levels, T)) ---
    def alloc(name: str, shape: tuple[int, int]) -> np.ndarray:
        if out_dir is None:
            return np.empty(shape, dtype=np.uint32)
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"{prefix}.{name}.u32.npy")
        if use_memmap:
            # Create memmap-backed .npy
            # We'll allocate via np.memmap on raw file, then save header is tricky;
            # simplest: use np.lib.format.open_memmap
            return np.lib.format.open_memmap(path, mode="w+", dtype=np.uint32, shape=shape)
        else:
            arr2 = np.empty(shape, dtype=np.uint32)
            np.save(path, arr2)
            return arr2

    hit_long_tp = alloc("long_tp", (n_tp, T))
    hit_long_sl = alloc("long_sl", (n_sl, T))
    hit_short_tp = alloc("short_tp", (n_tp, T))
    hit_short_sl = alloc("short_sl", (n_sl, T))

    # reuse heaps to avoid realloc per level
    heap_p = np.empty(T, dtype=np.float32)
    heap_i = np.empty(T, dtype=np.int32)
    out1d = np.empty(T, dtype=np.uint32)

    # --- warmup compile (tiny) ---
    _o = np.array([10, 10, 10, 10], dtype=np.float32)
    _h = np.array([10, 11, 10, 12], dtype=np.float32)
    _l = np.array([10,  9, 10,  8], dtype=np.float32)
    _hp = np.empty(_o.size, dtype=np.float32)
    _hi = np.empty(_o.size, dtype=np.int32)
    _out = np.empty(_o.size, dtype=np.uint32)
    _hit_long_tp_1level(_o, _h, np.float32(1.10), _out, _hp, _hi)
    _hit_long_sl_1level(_o, _l, np.float32(0.95), _out, _hp, _hi)
    _hit_short_tp_1level(_o, _l, np.float32(0.95), _out, _hp, _hi)
    _hit_short_sl_1level(_o, _h, np.float32(1.10), _out, _hp, _hi)

    # --- compute ---
    for k in range(n_tp):
        _hit_long_tp_1level(open_, high_, tp_mult_up[k], out1d, heap_p, heap_i)
        hit_long_tp[k, :] = out1d

    for k in range(n_sl):
        _hit_long_sl_1level(open_, low_, sl_mult_down[k], out1d, heap_p, heap_i)
        hit_long_sl[k, :] = out1d

    for k in range(n_tp):
        _hit_short_tp_1level(open_, low_, tp_mult_down[k], out1d, heap_p, heap_i)
        hit_short_tp[k, :] = out1d

    for k in range(n_sl):
        _hit_short_sl_1level(open_, high_, sl_mult_up[k], out1d, heap_p, heap_i)
        hit_short_sl[k, :] = out1d

    meta = {
        "T": int(T),
        "tp_grid_pct": tp_grid_pct.astype(np.float32),
        "sl_grid_pct": sl_grid_pct.astype(np.float32),
        "INF": int(T),
        "entry_rule": "entry_at_open[t]_inclusive_bar_t",
        "shape": {
            "hit_long_tp": tuple(hit_long_tp.shape),
            "hit_long_sl": tuple(hit_long_sl.shape),
            "hit_short_tp": tuple(hit_short_tp.shape),
            "hit_short_sl": tuple(hit_short_sl.shape),
        },
    }

    return {
        "meta": meta,
        "hit_long_tp": hit_long_tp,
        "hit_long_sl": hit_long_sl,
        "hit_short_tp": hit_short_tp,
        "hit_short_sl": hit_short_sl,
    }

In [4]:
npy_path = '/Users/daniildegtyarev/Projects/roehub.com/tests/notebook_tests/precompute/btcusdt_5m/prices_and_signals_5m.npy'
ohlc_5m_np = np.load(npy_path, allow_pickle=False)

In [7]:

hit = precompute_hit_times_execution_1m(
    ohlc_5m_np,
    open_col=2, high_col=3, low_col=4,
    tp_start_pct=0.5, tp_stop_pct=50.0, tp_step_pct=0.2,
    sl_start_pct=0.5, sl_stop_pct=25.0, sl_step_pct=0.2,
    out_dir="precompute/btcusdt_5m",   # можно None, если хочешь всё в RAM
    prefix="btc_5m",
    use_memmap=True,
)
